In [2]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp
from queue import Queue
from threading import Thread
from tqdm import tqdm
import math
import time

# -------------------------
# CONFIGURATION
# -------------------------

TILE_SIZE = 1024
BATCH_SIZE = 64
QUEUE_SIZE = 128
NDVI_THRESHOLD = 0.2

tile_queue = Queue(maxsize=QUEUE_SIZE)
result_queue = Queue(maxsize=QUEUE_SIZE)

total_tiles = 0
kept_tiles = 0

# -------------------------
# GPU Batch Processing
# -------------------------

def process_batch(batch_windows, batch_red, batch_nir):
    global kept_tiles

    red_stack = np.asarray(batch_red, dtype="float32") / 10000.0
    nir_stack = np.asarray(batch_nir, dtype="float32") / 10000.0

    red_gpu = cp.asarray(red_stack)
    nir_gpu = cp.asarray(nir_stack)

    ndvi_gpu = (nir_gpu - red_gpu) / (nir_gpu + red_gpu + 1e-6)

    # Mean NDVI per tile
    mean_ndvi = cp.mean(ndvi_gpu, axis=(1, 2))

    mask = mean_ndvi < NDVI_THRESHOLD

    ndvi_stack = cp.asnumpy(ndvi_gpu)

    for i in range(len(batch_windows)):
        if mask[i]:
            kept_tiles += 1
            result_queue.put((batch_windows[i], ndvi_stack[i]))

# -------------------------
# READER THREAD
# -------------------------

def reader(red_path, nir_path):
    global total_tiles

    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:
        h, w = red_src.height, red_src.width

        for r in range(0, h, TILE_SIZE):
            for c in range(0, w, TILE_SIZE):

                win_h = min(TILE_SIZE, h - r)
                win_w = min(TILE_SIZE, w - c)

                window = Window(c, r, win_w, win_h)

                red = red_src.read(
                    1,
                    window=window,
                    boundless=True,
                    fill_value=0,
                    out_shape=(TILE_SIZE, TILE_SIZE)
                )

                nir = nir_src.read(
                    1,
                    window=window,
                    boundless=True,
                    fill_value=0,
                    out_shape=(TILE_SIZE, TILE_SIZE)
                )

                tile_queue.put((window, red, nir))
                total_tiles += 1

    tile_queue.put(None)

# -------------------------
# WORKER THREAD
# -------------------------

def worker():
    batch_windows = []
    batch_red = []
    batch_nir = []

    while True:
        item = tile_queue.get()

        if item is None:
            if batch_windows:
                process_batch(batch_windows, batch_red, batch_nir)

            result_queue.put(None)
            break

        win, red, nir = item

        batch_windows.append(win)
        batch_red.append(red)
        batch_nir.append(nir)

        if len(batch_windows) == BATCH_SIZE:
            process_batch(batch_windows, batch_red, batch_nir)

            batch_windows = []
            batch_red = []
            batch_nir = []

# -------------------------
# WRITER THREAD
# -------------------------

def writer(output_path, profile, total_tiles_est):
    with rasterio.open(output_path, "w", **profile) as dst:

        pbar = tqdm(total=total_tiles_est, desc="Processing Tiles")

        while True:
            item = result_queue.get()

            if item is None:
                break

            win, ndvi = item

            dst.write(
                ndvi[:win.height, :win.width],
                1,
                window=win
            )

            pbar.update(1)

        pbar.close()

# -------------------------
# MAIN
# -------------------------

def main():

    red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
    nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
    output_path = "final_ndvi_filtered.tif"

    start = time.time()

    with rasterio.open(red_path) as src:
        h, w = src.height, src.width

        profile = src.profile
        profile.update(
            dtype="float32",
            count=1,
            driver="GTiff",
            tiled=True,
            compress="lzw"
        )

        total_tiles_est = math.ceil(h / TILE_SIZE) * math.ceil(w / TILE_SIZE)

    reader_thread = Thread(target=reader, args=(red_path, nir_path))
    worker_thread = Thread(target=worker)
    writer_thread = Thread(target=writer, args=(output_path, profile, total_tiles_est))

    reader_thread.start()
    worker_thread.start()
    writer_thread.start()

    reader_thread.join()
    worker_thread.join()
    writer_thread.join()

    elapsed = time.time() - start

    print("\n-------------------------------")
    print(f"Total tiles processed = {total_tiles}")
    print(f"Tiles kept (important) = {kept_tiles}")
    print(f"Compression ratio = {kept_tiles / total_tiles:.2f}")
    print(f"Total time = {elapsed:.2f} sec")
    print(f"Speed = {total_tiles / elapsed:.2f} tiles/sec")

# -------------------------

if __name__ == "__main__":
    main()

Processing Tiles:  84%|████████▍ | 102/121 [00:10<00:02,  9.29it/s]


-------------------------------
Total tiles processed = 121
Tiles kept (important) = 102
Compression ratio = 0.84
Total time = 11.10 sec
Speed = 10.90 tiles/sec
